# BirdCLEF 2026 - gold (submission-only)

Este notebook é uma versão corrigida para **inference/submission**.  
Ele **não treina** modelos durante a submissão.  
A ideia correta no Kaggle é:

1. Treinar em outro notebook.
2. Salvar os pesos (`.pth`) em um Dataset privado/público do Kaggle.
3. Anexar esse Dataset ao notebook de submissão.
4. Rodar apenas a inferência no conjunto oculto e gerar `submission.csv`.


In [ ]:

import os, glob, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch, torch.nn as nn
import torchaudio, timm
from tqdm import tqdm

# -----------------------------
# Paths
# -----------------------------
COMP_DIR_CANDIDATES = [
    '/kaggle/input/birdclef-2026',
    '/kaggle/input/competitions/birdclef-2026',
]

DATA = None
for p in COMP_DIR_CANDIDATES:
    if os.path.exists(os.path.join(p, 'sample_submission.csv')):
        DATA = p
        break

if DATA is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'sample_submission.csv' in files:
            DATA = root
            break

assert DATA is not None, 'Competition data not found.'

# IMPORTANT:
# Attach your model-weights dataset in Kaggle and point this path to it.
MODEL_DIR_CANDIDATES = [
    '/kaggle/input/birdclef2026-weights',
    '/kaggle/input/birdclef-2026-weights',
    '/kaggle/working/models',
]

MODELS_DIR = None
for p in MODEL_DIR_CANDIDATES:
    if os.path.exists(p):
        MODELS_DIR = p
        break

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SR = 32000
DUR = 5
AUDIO_LEN = SR * DUR
N_MELS = 128

sub_df = pd.read_csv(f'{DATA}/sample_submission.csv')
SPECIES = list(sub_df.columns)[1:]
NUM_CLASSES = len(SPECIES)

print('DATA =', DATA)
print('MODELS_DIR =', MODELS_DIR)
print('DEVICE =', DEVICE)
print('NUM_CLASSES =', NUM_CLASSES)


In [ ]:

class BirdModel(nn.Module):
    def __init__(self, backbone_name='tf_efficientnet_b0_ns'):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=False,
            num_classes=0,
            global_pool='avg'
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.backbone.num_features, NUM_CLASSES)
        )

    def forward(self, x):
        return self.head(self.backbone(x))

mel_tf = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR,
    n_fft=2048,
    hop_length=512,
    n_mels=N_MELS,
    f_min=20,
    f_max=16000
).to(DEVICE)

db_tf = torchaudio.transforms.AmplitudeToDB().to(DEVICE)


In [ ]:

weight_paths = []
if MODELS_DIR is not None:
    weight_paths = sorted(glob.glob(f'{MODELS_DIR}/**/best_*.pth', recursive=True))
    if not weight_paths:
        weight_paths = sorted(glob.glob(f'{MODELS_DIR}/**/*.pth', recursive=True))

print('Found weights:', len(weight_paths))
for p in weight_paths[:10]:
    print(' -', p)

ensemble = []
for path in weight_paths:
    ckpt = torch.load(path, map_location=DEVICE)
    backbone = ckpt.get('backbone', 'tf_efficientnet_b0_ns')
    model = BirdModel(backbone).to(DEVICE)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    score = float(ckpt.get('val_cmap', 1.0))
    ensemble.append((model, max(score, 1e-6)))
    print(f'Loaded {os.path.basename(path)} | backbone={backbone} | weight={score:.6f}')

if len(ensemble) == 0:
    print('No trained weights found. A dummy zero submission will be created.')

total_w = sum(w for _, w in ensemble) if ensemble else 1.0
print('Ensemble size =', len(ensemble))
print('Total weight =', total_w)


In [ ]:

def wav_to_spec_tensor(chunk_wav: np.ndarray) -> torch.Tensor:
    t = torch.tensor(chunk_wav, device=DEVICE, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    spec = db_tf(mel_tf(t)).expand(1, 3, -1, -1)
    spec = (spec - spec.mean()) / (spec.std() + 1e-6)
    return spec

test_files = sorted(glob.glob(f'{DATA}/test_soundscapes/*.ogg'))
print('Test soundscapes:', len(test_files))

if (len(test_files) == 0) or (len(ensemble) == 0):
    sub = pd.read_csv(f'{DATA}/sample_submission.csv')
    sub.iloc[:, 1:] = 0.0
    sub.to_csv('submission.csv', index=False)
    print('Created dummy submission.csv')
else:
    rows = []
    with torch.inference_mode():
        for fp in tqdm(test_files, desc='Inference'):
            prefix = os.path.basename(fp).replace('.ogg', '')
            wav, sr = torchaudio.load(fp)

            if sr != SR:
                wav = torchaudio.functional.resample(wav, sr, SR)

            if wav.shape[0] > 1:
                wav = wav.mean(0, keepdim=True)

            wav = wav[0].cpu().numpy()
            n_bins = max(1, int(np.ceil(len(wav) / AUDIO_LEN)))

            for i in range(n_bins):
                start = i * AUDIO_LEN
                chunk = wav[start:start + AUDIO_LEN]
                if len(chunk) < AUDIO_LEN:
                    chunk = np.pad(chunk, (0, AUDIO_LEN - len(chunk)))

                spec = wav_to_spec_tensor(chunk)
                pred = np.zeros(NUM_CLASSES, dtype=np.float32)

                for model, w in ensemble:
                    p = torch.sigmoid(model(spec)).cpu().numpy()[0].astype(np.float32)
                    pred += (w / total_w) * p

                row_id = f'{prefix}_{(i + 1) * 5}'
                rows.append([row_id] + pred.tolist())

    sub = pd.DataFrame(rows, columns=['row_id'] + SPECIES)
    sub.to_csv('submission.csv', index=False)
    print('submission.csv shape =', sub.shape)

print('Done.')
